## Dana Mounayer - S23108503

# CS4083 – NLP · Quiz: Text Pre-processing, BoW/TF-IDF & Vector Semantics
**Effat University · Fall 2026 · Instructor: Dr Passent Elkafrawy**

**Part B – Practical.** Put `quiz_reviews.csv`, `mini_corpus.txt` and `cooccurrence_counts.csv` in the same folder as this notebook. Write your code in the cells marked `# TODO`. Do not change random seeds.

Dataset columns: `id`, `text` (raw post), `label` (1 = positive, 0 = negative), `sentiment`, `source`.

In [1]:
# ---- Setup (given) ----
import re, string, math
from collections import Counter
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import regexp_tokenize, word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn import metrics

for pkg in ["stopwords", "punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

In [2]:
# ---- Given: emoticon / emoji dictionary (use instead of the emot library) ----
EMO_MAP = {
    ">:(": " angry ", ":'(": " crying ", ":-(": " sad ", ":(": " sad ",
    ":-)": " happy ", ":)": " happy ", "=)": " happy ", ":d": " laugh ", "xd": " laugh ",
    "😍": " love ", "😂": " laugh ", "👍": " thumbs_up ", "🎉": " celebrate ",
    "😡": " angry ", "😢": " crying ", "👎": " thumbs_down ",
}
# Negation words we must KEEP (same idea as the lab)
KEEP = {"no", "nor", "not", "don", "don't", "didn", "didn't", "doesn", "doesn't",
        "isn", "isn't", "wasn", "wasn't", "aren", "aren't", "won", "won't",
        "couldn", "couldn't", "shouldn", "shouldn't", "wouldn", "wouldn't"}

## B1. Load & explore the data (3 marks)
a) Load `quiz_reviews.csv`. Print its shape and the number of positive / negative posts.
b) How many rows have an **exact duplicate** `text`? Drop them (keep the first) and print the new shape.
c) Using the string method `find()` (as in the lab) on the **lower-cased** text, count how many posts contain `"http"`.

In [3]:
# TODO: write your code here
df = pd.read_csv("quiz_reviews.csv")
print("Shape:", df.shape)
print("Positive posts:", (df["label"] == 1).sum())
print("Negative posts:", (df["label"] == 0).sum())

n_dup = df.duplicated(subset="text").sum()
print("Rows with an exact duplicate text (extra copies):", n_dup)
df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
print("New shape after dropping duplicates:", df.shape)

n_http = sum(1 for t in df["text"] if t.lower().find("http") != -1)
print("Posts containing 'http':", n_http)


Shape: (506, 5)
Positive posts: 256
Negative posts: 250
Rows with an exact duplicate text (extra copies): 7
New shape after dropping duplicates: (499, 5)
Posts containing 'http': 93


The classes are almost balanced, so accuracy is a fair metric. Duplicates are removed so the same post can't appear in both train and test (data leakage).

## B2. Cleaning pipeline (5 marks)
Write `clean_text(t)` that applies these steps **in this order**:
1. lowercase  2. remove URLs with `r"http[s]?://\S+|www\.\S+"`  3. remove mentions `r"@\w+"`
4. replace every key of `EMO_MAP` with its word (replace **longer keys first**)  5. remove `#` but keep the hashtag word
6. remove every character that is not `a-z`, `_` or a space (digits and punctuation go)  7. collapse multiple spaces and strip
8. remove NLTK English stopwords **except** the words in `KEEP`.

Store the result in a new column `clean`. Print the original and cleaned text for `id` 4 and `id` 8, and the number of rows whose `clean` text contains the token `not`.

In [4]:
# TODO: write your code here
STOP = set(stopwords.words("english")) - KEEP
EMO_SORTED = sorted(EMO_MAP.items(), key=lambda kv: len(kv[0]), reverse=True)  # longer keys first

def clean_text(t):
    t = t.lower()                                              # 1. lowercase
    t = re.sub(r"http[s]?://\S+|www\.\S+", " ", t)             # 2. URLs
    t = re.sub(r"@\w+", " ", t)                                # 3. mentions
    for k, v in EMO_SORTED:                                    # 4. emoticons/emojis (longer first)
        t = t.replace(k, v)
    t = t.replace("#", "")                                     # 5. remove '#', keep the word
    t = re.sub(r"[^a-z_ ]", " ", t)                            # 6. keep only a-z, _ and space
    t = re.sub(r"\s+", " ", t).strip()                         # 7. collapse spaces
    t = " ".join(w for w in t.split() if w not in STOP)        # 8. stopwords (keep negations)
    return t

df["clean"] = df["text"].apply(clean_text)

for i in (4, 8):
    row = df[df["id"] == i].iloc[0]
    print(f"id {i}\n  original: {row['text']}\n  cleaned : {row['clean']}\n")

n_not = df["clean"].apply(lambda s: "not" in s.split()).sum()
print("Rows whose clean text contains the token 'not':", n_not)


id 4
  original: @sara_k didn't expect the new update to be this boring. what a waste of money
  cleaned : didn expect new update boring waste money

id 8
  original: Ruined my day, the gym doesn't work and isn't worth it >:(
  cleaned : ruined day gym doesn work isn worth angry

Rows whose clean text contains the token 'not': 98


Longer keys are replaced first so ">:(" becomes "angry" before ":(" can match part of it.

Removing the apostrophe (step 6) turns "didn't" into "didn t"; "t" is a stopword and is dropped, leaving "didn". That is why KEEP contains "didn", "doesn" and "isn": they hold the negation.

## B3. Tokenization, frequencies, stemming vs lemmatization (4 marks)
a) Tokenize `clean` with `regexp_tokenize(..., pattern=r"\s+", gaps=True)` into a column `tokens`. Report the mean and median number of tokens per post.
b) Use `collections.Counter` to list the **10 most frequent tokens**.
c) For the words `["disappointing", "services", "recommended", "better", "was", "studies"]` print a table: word | Porter stem | WordNet lemma (default) | WordNet lemma with `pos="v"`. Explain in one comment line why `was` behaves differently.

In [5]:
# TODO: write your code here
# a)
df["tokens"] = df["clean"].apply(lambda s: regexp_tokenize(s, pattern=r"\s+", gaps=True))
n_tok = df["tokens"].apply(len)
print("Mean tokens per post  :", round(n_tok.mean(), 3))
print("Median tokens per post:", n_tok.median())

# b)
counter = Counter(tok for toks in df["tokens"] for tok in toks)
print("\nTop 10 tokens:")
for w, c in counter.most_common(10):
    print(f"  {w:<15}{c}")

# c)
ps, wnl = PorterStemmer(), WordNetLemmatizer()
words = ["disappointing", "services", "recommended", "better", "was", "studies"]
tbl = pd.DataFrame({
    "word": words,
    "Porter stem": [ps.stem(w) for w in words],
    "Lemma (default)": [wnl.lemmatize(w) for w in words],
    "Lemma (pos='v')": [wnl.lemmatize(w, pos="v") for w in words],
})
print()
print(tbl.to_string(index=False))


Mean tokens per post  : 7.202
Median tokens per post: 7.0

Top 10 tokens:
  not            104
  new            94
  happy          79
  honestly       77
  didn           74
  expect         74
  service        68
  today          68
  phone          57
  worth          57

         word Porter stem Lemma (default) Lemma (pos='v')
disappointing  disappoint   disappointing      disappoint
     services      servic         service         service
  recommended   recommend     recommended       recommend
       better      better          better          better
          was          wa              wa              be
      studies       studi           study           study


'was' is an irregular verb form: the default POS is noun, so WordNet does not
treat it as a verb and only strips the "s" (giving "wa"); with pos="v" it looks up the dictionary and returns "be". A stemmer just chops suffixes by rules, so it also gives "wa".

Posts are short (about 7 tokens after cleaning). "not", "didn" and "expect" rank high because we kept negations. Stemming can produce non-words ("servic", "studi"); lemmatization returns real words but needs the correct POS tag ("better" would need pos="a" to become "good").

## B4. Bag-of-Words vs TF-IDF with Multinomial Naive Bayes (4 marks)
Use `X = df["clean"]`, `y = df["label"]` and **`train_test_split(test_size=0.3, random_state=42, stratify=y)`**.
a) `CountVectorizer(max_features=100, ngram_range=(1, 2))` → `MultinomialNB()`: report accuracy and the confusion matrix. List 5 bigram features that were kept.
b) `TfidfVectorizer()` (default settings) → `MultinomialNB()` on the same split: report accuracy.
c) In a comment: which one is better here and why (think about vocabulary size and IDF).

In [6]:
# TODO: write your code here
X = df["clean"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# a) BoW (max 100 features, uni+bigrams) + NB
cv = CountVectorizer(max_features=100, ngram_range=(1, 2))
Xtr_cv = cv.fit_transform(X_train)
Xte_cv = cv.transform(X_test)
nb_cv = MultinomialNB().fit(Xtr_cv, y_train)
pred_cv = nb_cv.predict(Xte_cv)
acc_cv = metrics.accuracy_score(y_test, pred_cv)
print("BoW accuracy:", round(acc_cv, 4))
print("Confusion matrix (rows = true 0/1, cols = predicted 0/1):\n", metrics.confusion_matrix(y_test, pred_cv))
bigrams = [f for f in cv.get_feature_names_out() if " " in f]
print("5 bigram features kept:", bigrams[:5])

# b) TF-IDF (defaults) + NB
tf = TfidfVectorizer()
Xtr_tf = tf.fit_transform(X_train)
Xte_tf = tf.transform(X_test)
nb_tf = MultinomialNB().fit(Xtr_tf, y_train)
acc_tf = metrics.accuracy_score(y_test, nb_tf.predict(Xte_tf))
print("TF-IDF accuracy:", round(acc_tf, 4), "| vocabulary size:", len(tf.vocabulary_))


BoW accuracy: 0.8467
Confusion matrix (rows = true 0/1, cols = predicted 0/1):
 [[60 14]
 [ 9 67]]
5 bigram features kept: ['best day', 'campus cafe', 'coffee shop', 'come back', 'coming back']
TF-IDF accuracy: 0.92 | vocabulary size: 128


c) TF-IDF is better here (0.92 vs 0.8467). The whole vocabulary is only ~128 words, so max_features=100 with bigrams throws away useful sentiment words (bigrams also compete for the 100 slots), while TF-IDF keeps all of them. IDF also down-weights very common words (like "not", "new", "today") that appear in
both classes, so the rarer, more discriminative words (e.g. "terrible", "amazing") count more.

In [7]:
# ---- Given: mini corpus + co-occurrence counts for Q B5 ----
docs = [line.strip() for line in open("mini_corpus.txt", encoding="utf-8") if line.strip()]
cooc = pd.read_csv("cooccurrence_counts.csv", index_col="target")
for i, d in enumerate(docs, 1):
    print(f"d{i}: {d}")
cooc

d1: the coffee was hot and the coffee was fresh
d2: the tea was cold and the service was slow
d3: fresh coffee and fresh bread make a great breakfast
d4: the service was great and the staff were friendly


,delicious,menu,battery,screen
target,,,,
coffee,40,30,2,3
pizza,35,45,1,4
phone,2,5,50,60
laptop,1,4,40,55


## B5. Vector semantics from scratch (4 marks)
Use the given `docs` (4 documents) and `cooc` (co-occurrence counts). **Do not use sklearn here — use plain Python / NumPy.**
a) Using the chapter's formula `tf = log10(count + 1)`, `idf = log10(N / df_t)`, compute TF-IDF of the terms `coffee`, `fresh`, `service`, `and` in each document (4×4 table, 3 decimals). Why is every value in the `and` column 0?
b) Build raw count vectors over the union vocabulary and compute **cosine similarity** for (d1, d3) and (d2, d4). Which pair is more similar?
c) Compute the **PPMI** matrix of `cooc` (log base 2). Which context word has the highest PPMI with `coffee`? With `phone`?
d) List the skip-gram **positive (target, context) pairs** for target `coffee` in d3 with a window of ±2.

In [8]:
# TODO: write your code here
# (uses docs and cooc from the given cell)
toks = [d.split() for d in docs]
N = len(toks)

# a) TF-IDF from scratch
terms = ["coffee", "fresh", "service", "and"]
df_t = {t: sum(1 for d in toks if t in d) for t in terms}
idf = {t: math.log10(N / df_t[t]) for t in terms}
tfidf = pd.DataFrame(
    {t: [math.log10(d.count(t) + 1) * idf[t] for d in toks] for t in terms},
    index=[f"d{i}" for i in range(1, N + 1)],
).round(3)
print("df:", df_t, "\nidf:", {k: round(v, 3) for k, v in idf.items()})
print(tfidf)
# 'and' appears in all 4 documents -> df = N -> idf = log10(4/4) = 0, so tf*idf = 0 in every document.
# A word that occurs everywhere has no power to discriminate between documents.

# b) Cosine similarity on raw count vectors
vocab = sorted(set(w for d in toks for w in d))
V = np.array([[d.count(w) for w in vocab] for d in toks], dtype=float)
def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
c13, c24 = cosine(V[0], V[2]), cosine(V[1], V[3])
print("\ncos(d1, d3) =", round(c13, 4))
print("cos(d2, d4) =", round(c24, 4))
print("More similar pair:", "(d1, d3)" if c13 > c24 else "(d2, d4)")

# c) PPMI (log2)
C = cooc.values.astype(float)
P = C / C.sum()
pw = P.sum(axis=1, keepdims=True)
pc = P.sum(axis=0, keepdims=True)
with np.errstate(divide="ignore"):
    pmi = np.log2(P / (pw * pc))
ppmi = pd.DataFrame(np.maximum(pmi, 0), index=cooc.index, columns=cooc.columns)
print("\nPPMI:\n", ppmi.round(3))
print("Highest PPMI with 'coffee':", ppmi.loc["coffee"].idxmax(), round(ppmi.loc["coffee"].max(), 3))
print("Highest PPMI with 'phone' :", ppmi.loc["phone"].idxmax(), round(ppmi.loc["phone"].max(), 3))

# d) Skip-gram positive pairs for 'coffee' in d3, window +-2
w = toks[2]
pairs = [(t, w[j]) for i, t in enumerate(w) if t == "coffee"
         for j in range(max(0, i - 2), min(len(w), i + 3)) if j != i]
print("\nd3:", w)
print("Positive pairs:", pairs)


df: {'coffee': 2, 'fresh': 2, 'service': 2, 'and': 4} 
idf: {'coffee': 0.301, 'fresh': 0.301, 'service': 0.301, 'and': 0.0}
    coffee  fresh  service  and
d1   0.144  0.091    0.000  0.0
d2   0.000  0.000    0.091  0.0
d3   0.091  0.144    0.000  0.0
d4   0.000  0.000    0.091  0.0

cos(d1, d3) = 0.3892
cos(d2, d4) = 0.669
More similar pair: (d2, d4)

PPMI:
         delicious   menu  battery  screen
target                                   
coffee      1.366  0.844    0.000   0.000
pizza       0.993  1.249    0.000   0.000
phone       0.000  0.000    0.793   0.664
laptop      0.000  0.000    0.697   0.765
Highest PPMI with 'coffee': delicious 1.366
Highest PPMI with 'phone' : battery 0.793

d3: ['fresh', 'coffee', 'and', 'fresh', 'bread', 'make', 'a', 'great', 'breakfast']
Positive pairs: [('coffee', 'fresh'), ('coffee', 'and'), ('coffee', 'fresh')]


b): d2 and d4 share "the", "was" (twice), "and" and "service", while d1 and d3 share only "coffee","fresh" and "and", and d3 is longer with many words d1 lacks.

c): PPMI is 0 for coffee/pizza with battery/screen and for phone/laptop with delicious/menu (negative PMI clipped to 0), so food words cluster together and tech words cluster together.

d): d3 = "fresh coffee and fresh bread ..."; coffee is at position 2, so the window covers fresh (left), and, fresh, (right). "bread" is 3 away, so it is outside the window.

## ⭐ B6 (EXTRA / BONUS – 5 marks). Train Word2vec on the quiz data
a) Train two `gensim` Word2Vec models on `df["tokens"]`: **skip-gram** (`sg=1`) and **CBOW** (`sg=0`), both with `vector_size=50, window=3, min_count=2, negative=5, epochs=50, seed=42, workers=1`.
b) For the skip-gram model print the 5 most similar words to `terrible` and to `amazing`, and the cosine similarity of (`great`, `awesome`) vs (`great`, `terrible`).
c) Build a document vector for every post as the **average (centroid)** of its word vectors (Chapter 6.7). Train `LogisticRegression(max_iter=1000)` on the same split as B4 and compare its accuracy with the TF-IDF model.
d) In a comment: give one reason why the embedding model may not beat TF-IDF on this dataset, and one limitation of static embeddings.

In [10]:
import sys
!{sys.executable} -m pip install gensim
# TODO: write your code here
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression

sentences = df["tokens"].tolist()
params = dict(vector_size=50, window=3, min_count=2, negative=5, epochs=50, seed=42, workers=1)
w2v_sg   = Word2Vec(sentences, sg=1, **params)   # skip-gram
w2v_cbow = Word2Vec(sentences, sg=0, **params)   # CBOW

# b)
for w in ["terrible", "amazing"]:
    print(f"\nMost similar to '{w}':")
    for word, s in w2v_sg.wv.most_similar(w, topn=5):
        print(f"  {word:<15}{s:.3f}")
print("\ncos(great, awesome)  =", round(float(w2v_sg.wv.similarity("great", "awesome")), 3))
print("cos(great, terrible) =", round(float(w2v_sg.wv.similarity("great", "terrible")), 3))

# c) centroid document vectors + Logistic Regression (same split as B4)
def doc_vec(tokens, wv=w2v_sg.wv):
    vecs = [wv[t] for t in tokens if t in wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size)

D = np.vstack(df["tokens"].apply(doc_vec).values)
idx_train, idx_test = train_test_split(df.index, test_size=0.3, random_state=42, stratify=df["label"])
lr = LogisticRegression(max_iter=1000).fit(D[idx_train], df.loc[idx_train, "label"])
acc_w2v = metrics.accuracy_score(df.loc[idx_test, "label"], lr.predict(D[idx_test]))
print("\nWord2vec-centroid + LogReg accuracy:", round(acc_w2v, 4))
print("TF-IDF + NB accuracy (B4)          :", round(acc_tf, 4))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 62.0 MB/s eta 0:00:00

Most similar to 'terrible':
  noisy          0.951
  stand          0.946
  boring         0.943
  rude           0.936
  expensive      0.935

Most similar to 'amazing':
  fast           0.963
  beautiful      0.950
  perfect        0.948
  wonderful      0.946
  brilliant      0.941

cos(great, awesome)  = 0.745
cos(great, terrible) = 0.602

Word2vec-centroid + LogReg accuracy: 0.8867
TF-IDF + NB accuracy (B4)          : 0.92


The neighbours of "terrible" are all negative words and those of "amazing" are positive, and (great, awesome) is more similar than (great, terrible), so the embeddings capture sentiment-related context.

d) Why it may not beat TF-IDF: the dataset is tiny (~500 short posts, ~127 words) so Word2vec has too little data to learn good vectors, and averaging word vectors loses word order and negation ("not good"), while TF-IDF + NB directly weights the discriminative words.

 Limitation of static embeddings: one vector per word regardless of context (polysemy, e.g. "cold" for coffee vs. a disease); also out-of-vocabulary words get no vector, and antonyms like good/bad end up close because they appear in similar contexts.